# Milestone 8 — Supervised Fine-Tuning

## Goal

Test whether supervised fine-tuning improves Ministral-3B's transferable
Text-to-SQL behavior beyond the context-engineered baseline.

## Experimental controls

- Base model: Ministral-3B
- Azure catalog version: 1
- Customization method: supervised fine-tuning
- Training type: Global
- Training records: 222
- Validation records: 85
- Frozen financial evaluation records: 24
- Frozen evaluation records are never used for training or model selection.
- The tuned and untuned models use the same inference and evaluation workflow.
- Only one initial fine-tuning job will be submitted.

## Initial hyperparameters

- Epochs: 2
- Seed: 42
- Batch size: automatic
- Learning-rate multiplier: automatic

Two epochs are intentionally conservative because the training set is small.
The objective is to learn reusable SQL behavior without memorizing the source
databases.

In [1]:
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import hashlib
import json
import os

from dotenv import load_dotenv


def find_repository_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


REPOSITORY_ROOT = find_repository_root(Path.cwd())
TRAINING_DIRECTORY = REPOSITORY_ROOT / "data" / "training"

load_dotenv(REPOSITORY_ROOT / ".env")

EXPECTED_ARTIFACTS = {
    "canonical": {
        "sha256": "7e2b0aeb4b5a00ba1b05665d803be7f5ea3b52cdedd87b5958e354c959eaa1a7",
    },
    "train_jsonl": {
        "sha256": "0b49016759ab6c07d16564f0d78a0ffefbac1b1eb0d2760b4fb6b3cbbcff65f8",
        "records": 222,
    },
    "validation_jsonl": {
        "sha256": "503be8d18732b1e3ff2df8e4e337af6ab8db971617c2fc49ed139f7a58fab25a",
        "records": 85,
    },
    "manifest": {
        "sha256": "9c98a0117ecb6003eda2a62aa93fab3f1bc464338d2d4cbb59f2240f7eb0a500",
    },
}

files_by_hash = {}

for path in TRAINING_DIRECTORY.rglob("*"):
    if path.is_file():
        files_by_hash[sha256_file(path)] = path

resolved_artifacts = {}

for artifact_name, expectation in EXPECTED_ARTIFACTS.items():
    expected_hash = expectation["sha256"]
    path = files_by_hash.get(expected_hash)

    if path is None:
        raise FileNotFoundError(
            f"Could not find {artifact_name} with SHA-256 "
            f"{expected_hash}"
        )

    resolved_artifacts[artifact_name] = path


def validate_sft_jsonl(
    path: Path,
    expected_records: int,
) -> Counter:
    raw_bytes = path.read_bytes()

    assert not raw_bytes.startswith(b"\xef\xbb\xbf"), (
        f"{path.name} contains a UTF-8 BOM"
    )

    role_sequences = Counter()
    record_count = 0

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            record = json.loads(line)

            messages = record.get("messages")
            assert isinstance(messages, list)
            assert messages

            roles = tuple(
                message.get("role")
                for message in messages
            )

            assert roles == ("system", "user", "assistant"), (
                f"Unexpected roles at line {line_number}: {roles}"
            )

            for message in messages:
                assert isinstance(message.get("content"), str)
                assert message["content"].strip()

            role_sequences[roles] += 1
            record_count += 1

    assert record_count == expected_records, (
        f"{path.name}: expected {expected_records} records, "
        f"found {record_count}"
    )

    return role_sequences


train_roles = validate_sft_jsonl(
    resolved_artifacts["train_jsonl"],
    EXPECTED_ARTIFACTS["train_jsonl"]["records"],
)

validation_roles = validate_sft_jsonl(
    resolved_artifacts["validation_jsonl"],
    EXPECTED_ARTIFACTS["validation_jsonl"]["records"],
)

project_endpoint = os.environ.get(
    "AZURE_AI_PROJECT_ENDPOINT",
    "",
).strip()

assert project_endpoint, (
    "AZURE_AI_PROJECT_ENDPOINT is missing."
)

assert "domain-adapt-project-resource" in project_endpoint, (
    "AZURE_AI_PROJECT_ENDPOINT does not appear to reference "
    "the new North Central US resource."
)

print("Fine-tuning input preflight passed.")
print("Project endpoint host:", urlparse(project_endpoint).netloc)
print(
    "Training artifact:",
    resolved_artifacts["train_jsonl"].relative_to(REPOSITORY_ROOT),
)
print(
    "Validation artifact:",
    resolved_artifacts["validation_jsonl"].relative_to(REPOSITORY_ROOT),
)
print("Training records:", sum(train_roles.values()))
print("Validation records:", sum(validation_roles.values()))
print("All expected artifact hashes matched.")
print("No model request or fine-tuning job was created.")

Fine-tuning input preflight passed.
Project endpoint host: domain-adapt-project-resource.services.ai.azure.com
Training artifact: data\training\sft_train_v1.jsonl
Validation artifact: data\training\sft_validation_v1.jsonl
Training records: 222
Validation records: 85
All expected artifact hashes matched.
No model request or fine-tuning job was created.


## Upload the frozen SFT artifacts

This step uploads the hash-verified training and validation JSONL files to
the new Foundry resource.

Uploading files does not start fine-tuning. The returned Azure file IDs are
recorded so that the eventual training job can be traced to the exact local
artifacts.

In [2]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

import datetime as dt


credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
)

openai_client = project_client.get_openai_client()

train_path = resolved_artifacts["train_jsonl"]
validation_path = resolved_artifacts["validation_jsonl"]


def upload_fine_tuning_file(path: Path):
    with path.open("rb") as handle:
        uploaded_file = openai_client.files.create(
            file=handle,
            purpose="fine-tune",
        )

    processed_file = openai_client.files.wait_for_processing(
        uploaded_file.id
    )

    if processed_file.status != "processed":
        raise RuntimeError(
            f"{path.name} was not processed successfully: "
            f"{processed_file.status}"
        )

    return processed_file


train_file = upload_fine_tuning_file(train_path)
validation_file = upload_fine_tuning_file(validation_path)

upload_receipt = {
    "created_at_utc": (
        dt.datetime.now(dt.timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    ),
    "project_endpoint_host": urlparse(project_endpoint).netloc,
    "training": {
        "local_path": str(
            train_path.relative_to(REPOSITORY_ROOT)
        ),
        "sha256": sha256_file(train_path),
        "records": 222,
        "azure_file_id": train_file.id,
        "azure_status": train_file.status,
        "bytes": train_file.bytes,
    },
    "validation": {
        "local_path": str(
            validation_path.relative_to(REPOSITORY_ROOT)
        ),
        "sha256": sha256_file(validation_path),
        "records": 85,
        "azure_file_id": validation_file.id,
        "azure_status": validation_file.status,
        "bytes": validation_file.bytes,
    },
}

receipt_directory = (
    REPOSITORY_ROOT
    / "data"
    / "runs"
    / "fine_tuning"
)
receipt_directory.mkdir(parents=True, exist_ok=True)

receipt_path = (
    receipt_directory
    / "ministral_3b_sft_v1_upload.json"
)

receipt_path.write_text(
    json.dumps(upload_receipt, indent=2) + "\n",
    encoding="utf-8",
)

print("Frozen SFT artifacts uploaded.")
print("Training file ID:", train_file.id)
print("Training status:", train_file.status)
print("Validation file ID:", validation_file.id)
print("Validation status:", validation_file.status)
print(
    "Receipt:",
    receipt_path.relative_to(REPOSITORY_ROOT),
)
print("No fine-tuning job was created.")

Frozen SFT artifacts uploaded.
Training file ID: file-a6c4da1ddcc24f9dab87afe6b76d711d
Training status: processed
Validation file ID: file-5e784e7e48b047e08793d7c6ffc80af3
Validation status: processed
Receipt: data\runs\fine_tuning\ministral_3b_sft_v1_upload.json
No fine-tuning job was created.


## Submit the controlled SFT Job

In [5]:
import hashlib
import json


upload_receipt = json.loads(
    receipt_path.read_text(encoding="utf-8")
)

training_file_id = upload_receipt[
    "training"
]["azure_file_id"]

validation_file_id = upload_receipt[
    "validation"
]["azure_file_id"]

training_file_status = openai_client.files.retrieve(
    training_file_id
)

validation_file_status = openai_client.files.retrieve(
    validation_file_id
)

assert training_file_status.status == "processed"
assert validation_file_status.status == "processed"

job_contract = {
    "experiment_name": "ministral_3b_sft_v1",
    "base_model": "Ministral-3B",
    "azure_catalog_version": "1",
    "customization_method": "supervised",
    "training_type": "GlobalStandard",
    "training_file_id": training_file_id,
    "training_sha256": upload_receipt[
        "training"
    ]["sha256"],
    "training_records": 222,
    "validation_file_id": validation_file_id,
    "validation_sha256": upload_receipt[
        "validation"
    ]["sha256"],
    "validation_records": 85,
    "n_epochs": 2,
    "seed": 42,
    "batch_size": "automatic",
    "learning_rate_multiplier": "automatic",
    "suffix": "domain-adapt-sft-v1",
}

canonical_contract = json.dumps(
    job_contract,
    sort_keys=True,
    separators=(",", ":"),
)

contract_sha256 = hashlib.sha256(
    canonical_contract.encode("utf-8")
).hexdigest()

print(json.dumps(job_contract, indent=2))
print()
print("Fine-tuning contract SHA-256:", contract_sha256)
print("Files are processed.")
print("No fine-tuning job created by this cell.")

{
  "experiment_name": "ministral_3b_sft_v1",
  "base_model": "Ministral-3B",
  "azure_catalog_version": "1",
  "customization_method": "supervised",
  "training_type": "GlobalStandard",
  "training_file_id": "file-ae7183a97bbc4590962362d034367fdc",
  "training_sha256": "0b49016759ab6c07d16564f0d78a0ffefbac1b1eb0d2760b4fb6b3cbbcff65f8",
  "training_records": 222,
  "validation_file_id": "file-0e2c62846e9f4ac9872f70205e2b7b6a",
  "validation_sha256": "503be8d18732b1e3ff2df8e4e337af6ab8db971617c2fc49ed139f7a58fab25a",
  "validation_records": 85,
  "n_epochs": 2,
  "seed": 42,
  "batch_size": "automatic",
  "learning_rate_multiplier": "automatic",
  "suffix": "domain-adapt-sft-v1"
}

Fine-tuning contract SHA-256: a8b01286ec4a0c39b581b29aea23e04cb57e7d1548126856740e6e4e5d02d5e1
Files are processed.
No fine-tuning job created by this cell.


In [6]:
SUBMIT_FINE_TUNING_JOB = True

job_receipt_path = (
    receipt_directory
    / "ministral_3b_sft_v1_job.json"
)

if not SUBMIT_FINE_TUNING_JOB:
    raise RuntimeError(
        "Submission is disabled."
    )

if job_receipt_path.exists():
    raise FileExistsError(
        "A job receipt already exists. Refusing to create "
        "a duplicate fine-tuning job."
    )

fine_tuning_job = openai_client.fine_tuning.jobs.create(
    training_file=training_file_id,
    validation_file=validation_file_id,
    model="Ministral-3B",
    suffix="domain-adapt-sft-v1",
    seed=42,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 2,
            }
        },
    },
    extra_body={
        "trainingType": "GlobalStandard",
    },
)

job_receipt = {
    "submitted_at_utc": (
        dt.datetime.now(dt.timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    ),
    "contract_sha256": contract_sha256,
    "contract": job_contract,
    "azure_job": fine_tuning_job.model_dump(
        mode="json"
    ),
}

job_receipt_path.write_text(
    json.dumps(job_receipt, indent=2) + "\n",
    encoding="utf-8",
)

print("Fine-tuning job created.")
print("Job ID:", fine_tuning_job.id)
print("Status:", fine_tuning_job.status)
print(
    "Receipt:",
    job_receipt_path.relative_to(REPOSITORY_ROOT),
)

Fine-tuning job created.
Job ID: ftjob-bd96ca624ae04994bab47135e9d12ebb
Status: pending
Receipt: data\runs\fine_tuning\ministral_3b_sft_v1_job.json


In [ ]:
import hashlib
import json


upload_receipt = json.loads(
    receipt_path.read_text(encoding="utf-8")
)

training_file_id = upload_receipt[
    "training"
]["azure_file_id"]

validation_file_id = upload_receipt[
    "validation"
]["azure_file_id"]

training_file_status = openai_client.files.retrieve(
    training_file_id
)

validation_file_status = openai_client.files.retrieve(
    validation_file_id
)

assert training_file_status.status == "processed"
assert validation_file_status.status == "processed"

job_contract = {
    "experiment_name": "ministral_3b_sft_v1",
    "base_model": "Ministral-3B",
    "azure_catalog_version": "1",
    "customization_method": "supervised",
    "training_type": "GlobalStandard",
    "training_file_id": training_file_id,
    "training_sha256": upload_receipt[
        "training"
    ]["sha256"],
    "training_records": 222,
    "validation_file_id": validation_file_id,
    "validation_sha256": upload_receipt[
        "validation"
    ]["sha256"],
    "validation_records": 85,
    "n_epochs": 2,
    "seed": 42,
    "batch_size": "automatic",
    "learning_rate_multiplier": "automatic",
    "suffix": "domain-adapt-sft-v1",
}

canonical_contract = json.dumps(
    job_contract,
    sort_keys=True,
    separators=(",", ":"),
)

contract_sha256 = hashlib.sha256(
    canonical_contract.encode("utf-8")
).hexdigest()

print(json.dumps(job_contract, indent=2))
print()
print("Fine-tuning contract SHA-256:", contract_sha256)
print("Files are processed.")
print("No fine-tuning job created by this cell.")

{
  "experiment_name": "ministral_3b_sft_v1",
  "base_model": "Ministral-3B",
  "azure_catalog_version": "1",
  "customization_method": "supervised",
  "training_type": "GlobalStandard",
  "training_file_id": "file-ae7183a97bbc4590962362d034367fdc",
  "training_sha256": "0b49016759ab6c07d16564f0d78a0ffefbac1b1eb0d2760b4fb6b3cbbcff65f8",
  "training_records": 222,
  "validation_file_id": "file-0e2c62846e9f4ac9872f70205e2b7b6a",
  "validation_sha256": "503be8d18732b1e3ff2df8e4e337af6ab8db971617c2fc49ed139f7a58fab25a",
  "validation_records": 85,
  "n_epochs": 2,
  "seed": 42,
  "batch_size": "automatic",
  "learning_rate_multiplier": "automatic",
  "suffix": "domain-adapt-sft-v1"
}

Fine-tuning contract SHA-256: a8b01286ec4a0c39b581b29aea23e04cb57e7d1548126856740e6e4e5d02d5e1
Files are processed.
No fine-tuning job created by this cell.


### Monitor the fine-tuning job

This cell retrieves the existing job and its recent events. It never creates,
restarts, or modifies a fine-tuning job.

A job normally moves through:

`pending → running → succeeded`

Terminal alternatives are `failed` or `cancelled`.

In [11]:
job_receipt = json.loads(
    job_receipt_path.read_text(encoding="utf-8")
)

job_id = job_receipt["azure_job"]["id"]

current_job = (
    openai_client
    .fine_tuning
    .jobs
    .retrieve(job_id)
)

events = (
    openai_client
    .fine_tuning
    .jobs
    .list_events(
        fine_tuning_job_id=job_id,
        limit=20,
    )
)

print("Job ID:", current_job.id)
print("Status:", current_job.status)
print(
    "Fine-tuned model:",
    current_job.fine_tuned_model,
)
print(
    "Trained tokens:",
    current_job.trained_tokens,
)
print("Error:", current_job.error)

print("\nRecent events:")

for event in reversed(events.data):
    print(
        f"- [{event.level}] {event.message}"
    )

status_snapshot = {
    "checked_at_utc": (
        dt.datetime.now(dt.timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    ),
    "job": current_job.model_dump(mode="json"),
    "events": [
        event.model_dump(mode="json")
        for event in events.data
    ],
}

status_path = (
    receipt_directory
    / "ministral_3b_sft_v1_status.json"
)

status_path.write_text(
    json.dumps(status_snapshot, indent=2) + "\n",
    encoding="utf-8",
)

print(
    "\nStatus snapshot:",
    status_path.relative_to(REPOSITORY_ROOT),
)
print("No model request or new training job created.")

Job ID: ftjob-bd96ca624ae04994bab47135e9d12ebb
Status: succeeded
Fine-tuned model: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-sft-v1
Trained tokens: 327712
Error: None

Recent events:
- [info] Job enqueued. Waiting for jobs ahead to complete.
- [info] Preprocessing running for file training file.
- [info] Preprocessing completed for file training file.
- [info] Preprocessing running for file validation file.
- [info] Preprocessing completed for file validation file.
- [info] Job started.
- [info] Finetuning started.
- [info] Created results file: file-3cee3a5393714929bfead0d96f8c13eb
- [info] Step 1: training loss=1.1993519067764282
- [info] Job succeeded.
- [info] Completed results file: file-3cee3a5393714929bfead0d96f8c13eb
- [info] Training tokens billed: 327830

Status snapshot: data\runs\fine_tuning\ministral_3b_sft_v1_status.json
No model request or new training job created.


### Download the training results

In [12]:
import csv
import io


completed_job = (
    openai_client
    .fine_tuning
    .jobs
    .retrieve(job_id)
)

assert completed_job.status == "succeeded"
assert completed_job.fine_tuned_model
assert completed_job.result_files

result_file_id = completed_job.result_files[0]
result_metadata = openai_client.files.retrieve(
    result_file_id
)

result_bytes = (
    openai_client
    .files
    .content(result_file_id)
    .read()
)

results_directory = (
    receipt_directory
    / "results"
)
results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

results_path = (
    results_directory
    / "ministral_3b_sft_v1_results.csv"
)

results_path.write_bytes(result_bytes)

results_sha256 = sha256_file(results_path)

print("Result file ID:", result_file_id)
print("Result filename:", result_metadata.filename)
print("Downloaded bytes:", len(result_bytes))
print("Results SHA-256:", results_sha256)
print(
    "Saved:",
    results_path.relative_to(REPOSITORY_ROOT),
)

Result file ID: file-3cee3a5393714929bfead0d96f8c13eb
Result filename: results.csv
Downloaded bytes: 668
Results SHA-256: eb2bcc1fcfb0729be4d5c8c9efba5181c21d975fd6767524d525ebdc942856a2
Saved: data\runs\fine_tuning\results\ministral_3b_sft_v1_results.csv


### Summarize the metrics

In [13]:
with results_path.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as handle:
    metric_rows = list(csv.DictReader(handle))

assert metric_rows, "The result file contains no metric rows."

metric_columns = list(metric_rows[0])

print("Metric rows:", len(metric_rows))
print("Metric columns:", metric_columns)

print("\nFirst metric row:")
print(json.dumps(metric_rows[0], indent=2))

print("\nFinal metric row:")
print(json.dumps(metric_rows[-1], indent=2))


def numeric_values(column_name):
    values = []

    for row in metric_rows:
        raw_value = row.get(column_name)

        if raw_value not in (None, ""):
            try:
                values.append(float(raw_value))
            except ValueError:
                pass

    return values


for metric_name in (
    "train_loss",
    "full_valid_loss",
    "valid_loss",
    "train_mean_token_accuracy",
    "full_valid_mean_token_accuracy",
    "valid_mean_token_accuracy",
):
    values = numeric_values(metric_name)

    if not values:
        continue

    print(
        f"\n{metric_name}:"
        f"\n  first: {values[0]:.6f}"
        f"\n  final: {values[-1]:.6f}"
        f"\n  minimum: {min(values):.6f}"
        f"\n  maximum: {max(values):.6f}"
    )

Metric rows: 1
Metric columns: ['step', 'train.allocated_mem', 'train.avg_wps', 'train.epochs_completed', 'train.eta_in_seconds', 'train.grad_norm', 'train_loss', 'train.lr', 'train.peak_allocated_mem', 'train.percent_done', 'train.samples_per_second', 'train.steps_per_second', 'train.total_flops', 'train.wps', 'train.train_runtime', 'eval.eval_loss', 'eval_loss', 'eval.eval_runtime', 'eval.eval_samples_per_second', 'eval.eval_steps_per_second', 'eval.perplexity', 'eval.train_loss']

First metric row:
{
  "step": "1",
  "train.allocated_mem": "17.006577014923096",
  "train.avg_wps": "3645.0911679410306",
  "train.epochs_completed": "1.7209519120301986",
  "train.eta_in_seconds": "0",
  "train.grad_norm": "1.5795339345932007",
  "train_loss": "1.1993519067764282",
  "train.lr": "3.9999999999999996E-10",
  "train.peak_allocated_mem": "55.7150444984436",
  "train.percent_done": "100",
  "train.samples_per_second": "7.008079332894438",
  "train.steps_per_second": "0.1112393544903879",
  "t

In [14]:
checkpoint_response = (
    openai_client
    .fine_tuning
    .jobs
    .checkpoints
    .list(job_id)
)

print(
    "Checkpoint count:",
    len(checkpoint_response.data),
)

for checkpoint in checkpoint_response.data:
    print("=" * 80)
    print(
        checkpoint.model_dump_json(
            indent=2
        )
    )

Checkpoint count: 0


## Verify the deployment

1. Verify the deployment with one smoke-test question.
2. Run all 12 working_dev questions.
3. If technically healthy, run the frozen 24 exactly once.
4. Compare against: Untuned context-engineered Ministral-3B Vs. Fine-tuned Ministral-3B

In [3]:
load_dotenv(
    REPOSITORY_ROOT / ".env",
    override=True,
)

sft_deployment_name = os.environ[
    "AZURE_SFT_MODEL_DEPLOYMENT_NAME"
].strip()

assert sft_deployment_name

print("SFT deployment:", sft_deployment_name)

connection_response = (
    openai_client
    .chat
    .completions
    .create(
        model=sft_deployment_name,
        messages=[
            {
                "role": "user",
                "content": (
                    "Reply with exactly: connection-ok"
                ),
            }
        ],
        max_tokens=10,
    )
)

connection_output = (
    connection_response
    .choices[0]
    .message
    .content
)

print("Response model:", connection_response.model)
print(
    "Finish reason:",
    connection_response.choices[0].finish_reason,
)
print("Output:", repr(connection_output))
print("Usage:", connection_response.usage)

SFT deployment: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-
Response model: ministral-3b-2410
Finish reason: stop
Output: 'connection-ok'
Usage: CompletionUsage(completion_tokens=3, prompt_tokens=9, total_tokens=12, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [4]:
from domain_adapt import context_engineering
import inspect
import json


public_functions = []

for name in dir(context_engineering):
    value = getattr(context_engineering, name)

    if (
        inspect.isfunction(value)
        and value.__module__
        == context_engineering.__name__
        and not name.startswith("_")
    ):
        public_functions.append(
            {
                "name": name,
                "signature": str(
                    inspect.signature(value)
                ),
            }
        )

print(
    json.dumps(
        public_functions,
        indent=2,
    )
)

[
  {
    "name": "load_context_engineering_config",
    "signature": "(path: str | pathlib._local.Path) -> dict[str, typing.Any]"
  },
  {
    "name": "request_context_engineered_sql",
    "signature": "(*, client: Any, deployment_name: str, question: str, config: dict[str, typing.Any]) -> tuple[typing.Any, str, str, bool]"
  },
  {
    "name": "unwrap_single_sql_fence",
    "signature": "(text: str | None) -> tuple[str, bool]"
  },
  {
    "name": "validate_context_engineering_config",
    "signature": "(config: dict[str, typing.Any]) -> None"
  }
]


### Question 16 controlled smoke test

In [5]:
from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_context_engineered_sql,
    validate_context_engineering_config,
)


config_candidates = list(
    REPOSITORY_ROOT.rglob(
        "slm_context_engineered_v1.json"
    )
)

assert len(config_candidates) == 1, (
    "Expected exactly one context-engineering config; "
    f"found {len(config_candidates)}"
)

context_config_path = config_candidates[0]
context_config_sha256 = sha256_file(
    context_config_path
)

assert context_config_sha256 == (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)

context_config = load_context_engineering_config(
    context_config_path
)

validate_context_engineering_config(
    context_config
)

question_id = 16
question = (
    "The transaction of 840 USD happened in "
    "1998/10/14, when was this account opened?"
)

(
    response,
    raw_sql,
    normalized_sql,
    output_envelope_normalized,
) = request_context_engineered_sql(
    client=openai_client,
    deployment_name=sft_deployment_name,
    question=question,
    config=context_config,
)

print("Question ID:", question_id)
print("Deployment:", sft_deployment_name)
print("Response model:", response.model)
print(
    "Finish reason:",
    response.choices[0].finish_reason,
)

print("\nRaw SQL:")
print(raw_sql)

print("\nNormalized SQL:")
print(normalized_sql)

print(
    "\nOutput envelope normalized:",
    output_envelope_normalized,
)

print("\nUsage:")
print(response.usage)

Question ID: 16
Deployment: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-
Response model: ministral-3b-2410
Finish reason: stop

Raw SQL:
SELECT account.date
FROM account
JOIN trans ON account.account_id = trans.account_id
WHERE trans.amount = 840 AND trans.date = '1998-10-14';

Normalized SQL:
SELECT account.date
FROM account
JOIN trans ON account.account_id = trans.account_id
WHERE trans.amount = 840 AND trans.date = '1998-10-14';

Output envelope normalized: False

Usage:
CompletionUsage(completion_tokens=44, prompt_tokens=1369, total_tokens=1413, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [6]:
import sqlite3


database_path = (
    REPOSITORY_ROOT
    / "data"
    / "financial.sqlite"
)

assert database_path.exists()

reference_sql = """
SELECT a.date
FROM trans AS t
INNER JOIN account AS a
    ON t.account_id = a.account_id
WHERE t.amount = 840
  AND t.date = '1998-10-14'
"""

database_uri = (
    f"file:{database_path.resolve().as_posix()}"
    "?mode=ro"
)

with sqlite3.connect(
    database_uri,
    uri=True,
) as connection:
    expected_rows = connection.execute(
        reference_sql
    ).fetchall()

    try:
        candidate_rows = connection.execute(
            normalized_sql
        ).fetchall()
        execution_error = None
    except sqlite3.Error as exc:
        candidate_rows = None
        execution_error = str(exc)

passed = (
    execution_error is None
    and candidate_rows == expected_rows
)

print("Expected rows:", expected_rows)
print("Candidate rows:", candidate_rows)
print("Execution error:", execution_error)
print("Passed:", passed)

Expected rows: [('1993-02-26',)]
Candidate rows: [('1993-02-26',)]
Execution error: None
Passed: True


### Run the 12-question working evaluation

In [7]:
# evaluation setup

from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from statistics import median
from time import perf_counter
import json
import os
import sqlite3

from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_context_engineered_sql,
)
from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible


DATA_DIR = Path("../data").resolve()
DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = Path(
    "../configs/eval_manifest_v1.json"
).resolve()
CONFIG_PATH = Path(
    "../configs/slm_context_engineered_v1.json"
).resolve()

WORKING_RUN_PATH = (
    DATA_DIR
    / "runs"
    / "fine_tuning"
    / "ministral_3b_sft_v1_working_dev.json"
)

EXPECTED_CONFIG_SHA256 = (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)

required_paths = (
    DB_PATH,
    DATASET_PATH,
    MANIFEST_PATH,
    CONFIG_PATH,
)

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        f"Missing prerequisites: {missing_paths}"
    )

assert sha256(
    CONFIG_PATH.read_bytes()
).hexdigest() == EXPECTED_CONFIG_SHA256

module_config = load_context_engineering_config(
    CONFIG_PATH
)

records = json.loads(
    DATASET_PATH.read_text(encoding="utf-8")
)
manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

working_ids = set(
    manifest["selection"]["working_dev"]["ids"]
)
frozen_ids = set(
    manifest["selection"]["frozen_eval"]["ids"]
)

assert working_ids.isdisjoint(frozen_ids)

working_examples = sorted(
    [
        record
        for record in records
        if record["db_name"] == "bird"
        and record["db_id"] == "financial"
        and record["question_id"] in working_ids
    ],
    key=lambda record: record["question_id"],
)

assert len(working_examples) == 12
assert {
    example["question_id"]
    for example in working_examples
} == working_ids

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True,
)

sft_deployment_name = os.environ[
    "AZURE_SFT_MODEL_DEPLOYMENT_NAME"
]

print("Working examples:", len(working_examples))
print("SFT deployment:", sft_deployment_name)
print("Frozen questions touched:", 0)
print("Configuration contract validated.")

Working examples: 12
SFT deployment: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-
Frozen questions touched: 0
Configuration contract validated.


In [9]:
def save_json_atomic(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary_path.write_text(
        json.dumps(value, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(path)


def is_executable(result: dict) -> bool:
    reason = result.get("reason") or ""

    return (
        result.get("api_error") is None
        and not reason.startswith(
            "candidate execution failed"
        )
        and not reason.startswith(
            "model request failed"
        )
    )


if WORKING_RUN_PATH.exists():
    working_run = json.loads(
        WORKING_RUN_PATH.read_text(
            encoding="utf-8"
        )
    )

    # Refuse to resume a run made under another contract.
    assert working_run["split"] == "working_dev"
    assert working_run["config_sha256"] == (
        EXPECTED_CONFIG_SHA256
    )
    assert working_run["deployment_name"] == (
        sft_deployment_name
    )
    assert working_run["attempts_per_question"] == 1
    assert working_run["candidate_sql_repair"] is False

else:
    working_run = {
        "run_version": 1,
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "benchmark": manifest["benchmark"],
        "split": "working_dev",
        "variant": "ministral-3b-sft-v1",
        "deployment_name": sft_deployment_name,
        "config_sha256": EXPECTED_CONFIG_SHA256,
        "instructions_sha256": (
            module_config["instructions_sha256"]
        ),
        "schema_context_sha256": (
            module_config["schema_context_sha256"]
        ),
        "attempts_per_question": 1,
        "candidate_sql_repair": False,
        "retry_incorrect_answers": False,
        "output_normalization": (
            module_config["output_normalization"]
        ),
        "results": [],
    }

    save_json_atomic(
        WORKING_RUN_PATH,
        working_run,
    )


completed_ids = {
    result["question_id"]
    for result in working_run["results"]
}

assert completed_ids.issubset(working_ids)
assert len(completed_ids) == len(
    working_run["results"]
)

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

print("Already completed:", len(completed_ids))
print("Pending:", len(pending_examples))
print("Artifact:", WORKING_RUN_PATH)

Already completed: 12
Pending: 0
Artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\fine_tuning\ministral_3b_sft_v1_working_dev.json


In [10]:
# Change only this assignment if your successful client object
# has a different variable name.
evaluation_client = openai_client

In [11]:
# Recover the last clean, saved state.
working_run = json.loads(
    WORKING_RUN_PATH.read_text(encoding="utf-8")
)

saved_ids = [
    result["question_id"]
    for result in working_run["results"]
]

assert len(saved_ids) == len(set(saved_ids)), (
    "The saved artifact itself contains duplicate IDs."
)

completed_ids = set(saved_ids)

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

evaluation_client = openai_client

print("Preserved completed results:", len(completed_ids))
print("Remaining questions:", len(pending_examples))
print("Completed IDs:", sorted(completed_ids))

Preserved completed results: 12
Remaining questions: 0
Completed IDs: [5, 11, 14, 16, 45, 49, 61, 69, 81, 89, 99, 105]


In [12]:
for example in working_examples:
    question_id = example["question_id"]

    # Recompute from current results so rerunning this cell is safe.
    completed_ids = {
        result["question_id"]
        for result in working_run["results"]
    }

    if question_id in completed_ids:
        continue

    assert question_id in working_ids
    assert question_id not in frozen_ids

    started = perf_counter()

    try:
        (
            response,
            raw_sql,
            normalized_sql,
            was_unwrapped,
        ) = request_context_engineered_sql(
            client=evaluation_client,
            deployment_name=sft_deployment_name,
            question=example["question"],
            config=module_config,
        )

        reference_sql, applied_rules = (
            make_reference_compatible(
                example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=reference_sql,
            candidate_sql=normalized_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": question_id,
            "difficulty": example["difficulty"],
            "raw_generated_sql": raw_sql,
            "generated_sql": normalized_sql,
            "output_envelope_normalized": was_unwrapped,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": (
                comparison.expected_row_count
            ),
            "candidate_row_count": (
                comparison.candidate_row_count
            ),
            "reference_compatibility_rules": (
                applied_rules
            ),
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        result = {
            "question_id": question_id,
            "difficulty": example["difficulty"],
            "raw_generated_sql": None,
            "generated_sql": None,
            "output_envelope_normalized": False,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "usage": {},
            "api_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    result["latency_seconds"] = round(
        perf_counter() - started,
        3,
    )

    # Guard immediately before mutation.
    assert question_id not in {
        item["question_id"]
        for item in working_run["results"]
    }

    working_run["results"].append(result)

    # Save immediately after each request.
    save_json_atomic(
        WORKING_RUN_PATH,
        working_run,
    )

    completed_count = len(
        working_run["results"]
    )
    outcome = (
        "PASS" if result["passed"] else "FAIL"
    )

    print(
        f"[{completed_count:02d}/12] "
        f"ID {question_id:>3} "
        f"({example['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

print(
    "Completed:",
    len(working_run["results"]),
    "/ 12",
)

Completed: 12 / 12


In [13]:
results = working_run["results"]

assert len(results) == 12
assert {
    result["question_id"]
    for result in results
} == working_ids

passed_count = sum(
    result["passed"]
    for result in results
)

executable_count = sum(
    is_executable(result)
    for result in results
)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(
            result["passed"]
            for result in subset
        ),
        "total": len(subset),
        "accuracy": (
            sum(
                result["passed"]
                for result in subset
            ) / len(subset)
            if subset
            else None
        ),
    }


def failure_category(result: dict) -> str:
    if result.get("api_error") is not None:
        return "api_error"

    reason = result.get("reason") or ""

    if reason.startswith(
        "candidate execution failed"
    ):
        return "candidate_execution_failed"

    if reason.startswith(
        "column count differs"
    ):
        return "column_count_differs"

    if reason == "ordered results differ":
        return "ordered_results_differ"

    if reason == "unordered results differ":
        return "unordered_results_differ"

    return reason.replace(" ", "_")


failure_categories = Counter(
    failure_category(result)
    for result in results
    if not result["passed"]
)

token_usage = {
    token_name: sum(
        result.get("usage", {}).get(
            token_name,
            0,
        )
        for result in results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

latencies = [
    result["latency_seconds"]
    for result in results
]

summary = {
    "passed": passed_count,
    "total": len(results),
    "accuracy": passed_count / len(results),
    "executable": executable_count,
    "executable_rate": (
        executable_count / len(results)
    ),
    "by_difficulty": difficulty_summary,
    "failure_categories": dict(
        failure_categories
    ),
    "api_errors": sum(
        result.get("api_error") is not None
        for result in results
    ),
    "strictly_unwrapped_outputs": sum(
        result.get(
            "output_envelope_normalized",
            False,
        )
        for result in results
    ),
    "observed_response_models": sorted({
        result["response_model"]
        for result in results
        if result.get("response_model")
    }),
    "token_usage": token_usage,
    "median_latency_seconds": median(
        latencies
    ),
}

working_run["summary"] = summary

save_json_atomic(
    WORKING_RUN_PATH,
    working_run,
)

run_sha256 = sha256(
    WORKING_RUN_PATH.read_bytes()
).hexdigest()

print("Fine-tuned working evaluation complete.")
print(
    f"Overall: {passed_count}/{len(results)} "
    f"({passed_count / len(results):.1%})"
)
print(
    f"Executable: {executable_count}/{len(results)} "
    f"({executable_count / len(results):.1%})"
)
print("By difficulty:", difficulty_summary)
print(
    "Failure categories:",
    dict(failure_categories),
)
print("API errors:", summary["api_errors"])
print(
    "Strictly unwrapped outputs:",
    summary["strictly_unwrapped_outputs"],
)
print(
    "Observed response models:",
    summary["observed_response_models"],
)
print("Token usage:", token_usage)
print(
    "Median latency seconds:",
    summary["median_latency_seconds"],
)
print("Working-run SHA-256:", run_sha256)
print("Artifact:", WORKING_RUN_PATH)

Fine-tuned working evaluation complete.
Overall: 0/12 (0.0%)
Executable: 0/12 (0.0%)
By difficulty: {'easy': {'passed': 0, 'total': 6, 'accuracy': 0.0}, 'medium': {'passed': 0, 'total': 4, 'accuracy': 0.0}, 'hard': {'passed': 0, 'total': 2, 'accuracy': 0.0}}
Failure categories: {'api_error': 12}
API errors: 12
Strictly unwrapped outputs: 0
Observed response models: []
Token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}
Median latency seconds: 0.0
Working-run SHA-256: 0cdb63028fc1ae5485210b661a5f226b8e75c9dd52e99e1dd74c52746d95339f
Artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\fine_tuning\ministral_3b_sft_v1_working_dev.json


## New attempt

In [14]:
from collections import Counter

recorded_errors = Counter(
    result["api_error"]
    for result in working_run["results"]
)

for error, count in recorded_errors.items():
    print("=" * 80)
    print("Count:", count)
    print("Error:", error)

Count: 12
Error: NameError: name 'evaluation_client' is not defined


In [15]:
infrastructure_only = [
    result
    for result in working_run["results"]
    if (
        result["api_error"] is not None
        and result["response_id"] is None
        and result["generated_sql"] is None
        and not result["usage"]
    )
]

print(
    "Infrastructure-only failures:",
    len(infrastructure_only),
)

Infrastructure-only failures: 12


In [16]:
FAILED_RUN_PATH = WORKING_RUN_PATH

WORKING_RUN_PATH = (
    DATA_DIR
    / "runs"
    / "fine_tuning"
    / "ministral_3b_sft_v1_working_dev_v2.json"
)

assert not WORKING_RUN_PATH.exists(), (
    f"Refusing to overwrite: {WORKING_RUN_PATH}"
)

evaluation_client = openai_client

# Verify the exact deployment variable again.
sft_deployment_name = os.environ[
    "AZURE_SFT_MODEL_DEPLOYMENT_NAME"
]

print("Deployment:", sft_deployment_name)
print("Client type:", type(evaluation_client))
print("New artifact:", WORKING_RUN_PATH)

Deployment: Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-
Client type: <class 'openai.OpenAI'>
New artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\fine_tuning\ministral_3b_sft_v1_working_dev_v2.json


In [17]:
connectivity_response = (
    evaluation_client.chat.completions.create(
        model=sft_deployment_name,
        messages=[
            {
                "role": "user",
                "content": (
                    "Reply with exactly: connection-ok"
                ),
            }
        ],
        max_tokens=10,
    )
)

print(
    "Output:",
    repr(
        connectivity_response
        .choices[0]
        .message
        .content
    ),
)
print(
    "Response model:",
    connectivity_response.model,
)
print(
    "Usage:",
    connectivity_response.usage,
)

Output: 'connection-ok'
Response model: ministral-3b-2410
Usage: CompletionUsage(completion_tokens=3, prompt_tokens=9, total_tokens=12, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [18]:
working_run = {
    "run_version": 2,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "benchmark": manifest["benchmark"],
    "split": "working_dev",
    "variant": "ministral-3b-sft-v1",
    "deployment_name": sft_deployment_name,
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "instructions_sha256": (
        module_config["instructions_sha256"]
    ),
    "schema_context_sha256": (
        module_config["schema_context_sha256"]
    ),
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "output_normalization": (
        module_config["output_normalization"]
    ),
    "supersedes_failed_run": (
        FAILED_RUN_PATH.name
    ),
    "superseded_failure_reason": (
        "All records failed before receiving a model response."
    ),
    "results": [],
}

save_json_atomic(
    WORKING_RUN_PATH,
    working_run,
)

print("Initialized empty v2 run.")
print("Results:", len(working_run["results"]))

Initialized empty v2 run.
Results: 0


In [19]:
# Recover the last clean, saved state.
working_run = json.loads(
    WORKING_RUN_PATH.read_text(encoding="utf-8")
)

saved_ids = [
    result["question_id"]
    for result in working_run["results"]
]

assert len(saved_ids) == len(set(saved_ids)), (
    "The saved artifact itself contains duplicate IDs."
)

completed_ids = set(saved_ids)

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

evaluation_client = openai_client

print("Preserved completed results:", len(completed_ids))
print("Remaining questions:", len(pending_examples))
print("Completed IDs:", sorted(completed_ids))

Preserved completed results: 0
Remaining questions: 12
Completed IDs: []


In [20]:
for example in working_examples:
    question_id = example["question_id"]

    # Recompute from current results so rerunning this cell is safe.
    completed_ids = {
        result["question_id"]
        for result in working_run["results"]
    }

    if question_id in completed_ids:
        continue

    assert question_id in working_ids
    assert question_id not in frozen_ids

    started = perf_counter()

    try:
        (
            response,
            raw_sql,
            normalized_sql,
            was_unwrapped,
        ) = request_context_engineered_sql(
            client=evaluation_client,
            deployment_name=sft_deployment_name,
            question=example["question"],
            config=module_config,
        )

        reference_sql, applied_rules = (
            make_reference_compatible(
                example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=reference_sql,
            candidate_sql=normalized_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": question_id,
            "difficulty": example["difficulty"],
            "raw_generated_sql": raw_sql,
            "generated_sql": normalized_sql,
            "output_envelope_normalized": was_unwrapped,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": (
                comparison.expected_row_count
            ),
            "candidate_row_count": (
                comparison.candidate_row_count
            ),
            "reference_compatibility_rules": (
                applied_rules
            ),
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        result = {
            "question_id": question_id,
            "difficulty": example["difficulty"],
            "raw_generated_sql": None,
            "generated_sql": None,
            "output_envelope_normalized": False,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "usage": {},
            "api_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    result["latency_seconds"] = round(
        perf_counter() - started,
        3,
    )

    # Guard immediately before mutation.
    assert question_id not in {
        item["question_id"]
        for item in working_run["results"]
    }

    working_run["results"].append(result)

    # Save immediately after each request.
    save_json_atomic(
        WORKING_RUN_PATH,
        working_run,
    )

    completed_count = len(
        working_run["results"]
    )
    outcome = (
        "PASS" if result["passed"] else "FAIL"
    )

    print(
        f"[{completed_count:02d}/12] "
        f"ID {question_id:>3} "
        f"({example['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

print(
    "Completed:",
    len(working_run["results"]),
    "/ 12",
)

[01/12] ID   5 (medium) PASS: unordered results match
[02/12] ID  11 (medium) FAIL: ordered results differ
[03/12] ID  14 (easy  ) FAIL: column count differs: 1 != 2
[04/12] ID  16 (easy  ) PASS: unordered results match
[05/12] ID  45 (easy  ) FAIL: ordered results differ
[06/12] ID  49 (medium) FAIL: unordered results differ
[07/12] ID  61 (hard  ) FAIL: unordered results differ
[08/12] ID  69 (easy  ) PASS: unordered results match
[09/12] ID  81 (hard  ) FAIL: candidate execution failed: no such column: account.account_id
[10/12] ID  89 (easy  ) FAIL: candidate execution failed: no such column: account.client_id
[11/12] ID  99 (easy  ) PASS: unordered results match
[12/12] ID 105 (medium) FAIL: candidate execution failed: ambiguous column name: client_id
Completed: 12 / 12


In [21]:
results = working_run["results"]

assert len(results) == 12
assert {
    result["question_id"]
    for result in results
} == working_ids

passed_count = sum(
    result["passed"]
    for result in results
)

executable_count = sum(
    is_executable(result)
    for result in results
)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(
            result["passed"]
            for result in subset
        ),
        "total": len(subset),
        "accuracy": (
            sum(
                result["passed"]
                for result in subset
            ) / len(subset)
            if subset
            else None
        ),
    }


def failure_category(result: dict) -> str:
    if result.get("api_error") is not None:
        return "api_error"

    reason = result.get("reason") or ""

    if reason.startswith(
        "candidate execution failed"
    ):
        return "candidate_execution_failed"

    if reason.startswith(
        "column count differs"
    ):
        return "column_count_differs"

    if reason == "ordered results differ":
        return "ordered_results_differ"

    if reason == "unordered results differ":
        return "unordered_results_differ"

    return reason.replace(" ", "_")


failure_categories = Counter(
    failure_category(result)
    for result in results
    if not result["passed"]
)

token_usage = {
    token_name: sum(
        result.get("usage", {}).get(
            token_name,
            0,
        )
        for result in results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

latencies = [
    result["latency_seconds"]
    for result in results
]

summary = {
    "passed": passed_count,
    "total": len(results),
    "accuracy": passed_count / len(results),
    "executable": executable_count,
    "executable_rate": (
        executable_count / len(results)
    ),
    "by_difficulty": difficulty_summary,
    "failure_categories": dict(
        failure_categories
    ),
    "api_errors": sum(
        result.get("api_error") is not None
        for result in results
    ),
    "strictly_unwrapped_outputs": sum(
        result.get(
            "output_envelope_normalized",
            False,
        )
        for result in results
    ),
    "observed_response_models": sorted({
        result["response_model"]
        for result in results
        if result.get("response_model")
    }),
    "token_usage": token_usage,
    "median_latency_seconds": median(
        latencies
    ),
}

working_run["summary"] = summary

save_json_atomic(
    WORKING_RUN_PATH,
    working_run,
)

run_sha256 = sha256(
    WORKING_RUN_PATH.read_bytes()
).hexdigest()

print("Fine-tuned working evaluation complete.")
print(
    f"Overall: {passed_count}/{len(results)} "
    f"({passed_count / len(results):.1%})"
)
print(
    f"Executable: {executable_count}/{len(results)} "
    f"({executable_count / len(results):.1%})"
)
print("By difficulty:", difficulty_summary)
print(
    "Failure categories:",
    dict(failure_categories),
)
print("API errors:", summary["api_errors"])
print(
    "Strictly unwrapped outputs:",
    summary["strictly_unwrapped_outputs"],
)
print(
    "Observed response models:",
    summary["observed_response_models"],
)
print("Token usage:", token_usage)
print(
    "Median latency seconds:",
    summary["median_latency_seconds"],
)
print("Working-run SHA-256:", run_sha256)
print("Artifact:", WORKING_RUN_PATH)

Fine-tuned working evaluation complete.
Overall: 4/12 (33.3%)
Executable: 9/12 (75.0%)
By difficulty: {'easy': {'passed': 3, 'total': 6, 'accuracy': 0.5}, 'medium': {'passed': 1, 'total': 4, 'accuracy': 0.25}, 'hard': {'passed': 0, 'total': 2, 'accuracy': 0.0}}
Failure categories: {'ordered_results_differ': 2, 'column_count_differs': 1, 'unordered_results_differ': 2, 'candidate_execution_failed': 3}
API errors: 0
Strictly unwrapped outputs: 5
Observed response models: ['ministral-3b-2410']
Token usage: {'prompt_tokens': 16406, 'completion_tokens': 2648, 'total_tokens': 19054}
Median latency seconds: 0.938
Working-run SHA-256: 23cc1ffe2db857ea418d70518082c3ab782a3d15cc1f6a6c27653c6a0423297b
Artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\fine_tuning\ministral_3b_sft_v1_working_dev_v2.json


In [22]:
BASELINE_RUN_PATH = (
    DATA_DIR
    / "runs"
    / "ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json"
)

SFT_RUN_PATH = (
    DATA_DIR
    / "runs"
    / "fine_tuning"
    / "ministral_3b_sft_v1_working_dev_v2.json"
)

baseline_run = json.loads(
    BASELINE_RUN_PATH.read_text(encoding="utf-8")
)
sft_run = json.loads(
    SFT_RUN_PATH.read_text(encoding="utf-8")
)

baseline_by_id = {
    result["question_id"]: result
    for result in baseline_run["results"]
}

sft_by_id = {
    result["question_id"]: result
    for result in sft_run["results"]
}

print(
    "ID | Base pass/exec | SFT pass/exec | Transition"
)
print("-" * 61)

for question_id in sorted(working_ids):
    baseline = baseline_by_id[question_id]
    tuned = sft_by_id[question_id]

    baseline_pass = baseline["passed"]
    tuned_pass = tuned["passed"]

    if not baseline_pass and tuned_pass:
        transition = "IMPROVED"
    elif baseline_pass and not tuned_pass:
        transition = "REGRESSED"
    elif tuned_pass:
        transition = "STILL PASS"
    else:
        transition = "STILL FAIL"

    print(
        f"{question_id:>3} | "
        f"{str(baseline_pass):<5}/"
        f"{str(is_executable(baseline)):<5} | "
        f"{str(tuned_pass):<5}/"
        f"{str(is_executable(tuned)):<5} | "
        f"{transition}"
    )

ID | Base pass/exec | SFT pass/exec | Transition
-------------------------------------------------------------
  5 | True /True  | True /True  | STILL PASS
 11 | False/False | False/True  | STILL FAIL
 14 | False/True  | False/True  | STILL FAIL
 16 | True /True  | True /True  | STILL PASS
 45 | False/True  | False/True  | STILL FAIL
 49 | False/True  | False/True  | STILL FAIL
 61 | False/True  | False/True  | STILL FAIL
 69 | True /True  | True /True  | STILL PASS
 81 | False/False | False/False | STILL FAIL
 89 | False/False | False/False | STILL FAIL
 99 | True /True  | True /True  | STILL PASS
105 | False/False | False/False | STILL FAIL


In [23]:
for question_id in sorted(working_ids):
    baseline = baseline_by_id[question_id]
    tuned = sft_by_id[question_id]

    if tuned["passed"]:
        continue

    print("=" * 88)
    print("Question ID:", question_id)
    print("Difficulty:", tuned["difficulty"])
    print()
    print("BASELINE SQL:")
    print(baseline.get("generated_sql"))
    print("Baseline result:", baseline["reason"])
    print()
    print("FINE-TUNED SQL:")
    print(tuned.get("generated_sql"))
    print("Fine-tuned result:", tuned["reason"])

Question ID: 11
Difficulty: medium

BASELINE SQL:
SELECT account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;
Baseline result: candidate execution failed: ambiguous column name: account_id

FINE-TUNED SQL:
SELECT account.account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
  AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;
Fine-tuned result: ordered results differ
Question ID: 14
Difficulty: easy

BASELINE SQL:
SELECT DISTINCT account.account_id, account.date
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE account.date < '1997-01-01'
  AND loan.amount > 3000;
Baseline result: column count differs: 1 != 2

FINE-TUNED SQL:
SELECT DISTINCT account.account_id, account.date
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE account.date < '1997-01-01'
AND loan.amount > 3000;
Fin